# Credit Scoring Model - Example Notebook

This notebook demonstrates how to use the Credit Scoring Model pipeline for predicting creditworthiness.

## Project Structure
```
Credit_Scoring_Model/
├── config/
│   └── config.yaml          # Configuration file
├── src/
│   ├── config.py            # Configuration management
│   ├── logging_config.py    # Logging setup
│   ├── data/
│   │   └── preprocessing.py # Data loading & preprocessing
│   ├── features/
│   │   └── feature_engineering.py # Feature engineering
│   ├── models/
│   │   └── train.py         # Model training with hyperparameter tuning
│   ├── evaluation/
│   │   └── evaluator.py     # Model evaluation & visualization
│   └── pipeline/
│       └── run_pipeline.py  # Main pipeline runner
├── scripts/
│   ├── generate_data.py     # Generate sample data
│   └── test_pipeline.py     # Test script
├── data/
│   ├── raw/                 # Raw data files
│   └── processed/           # Processed data
├── models/                  # Saved models
├── results/
│   ├── plots/               # Evaluation plots
│   └── metrics.json         # Metrics output
├── logs/                    # Log files
├── requirements.txt
└── main.py                  # CLI entry point
```

In [ ]:
# Install requirements (run once)
!pip install -q -r requirements.txt

# Add src to path
import sys
sys.path.insert(0, '../src')

## Quick Start

Run the full pipeline with default settings:

In [ ]:
from pipeline.run_pipeline import run_pipeline

# Run pipeline with sample data (will generate German Credit-like data)
results = run_pipeline()

print(f"Best Model: {results['summary']['best_model']}")
print(f"Test ROC-AUC: {results['summary']['test_roc_auc']:.4f}")
print(f"Test F1: {results['summary']['test_f1']:.4f}")

## Using Your Own Data

Prepare a CSV file with your credit data. The target column should be named `credit_risk` (configurable in config.yaml) with values `good`/`bad` or `1`/`0`.

In [ ]:
results = run_pipeline(
    data_path="data/raw/your_credit_data.csv",
    models_to_train=["logistic_regression", "random_forest", "xgboost", "lightgbm"],
    tune_hyperparams=True,
    output_dir="results/my_experiment"
)

## Making Predictions on New Data

In [ ]:
from pipeline.run_pipeline import predict
import pandas as pd

# Load new data
new_data = pd.read_csv("data/raw/new_applications.csv")

# Make predictions
predictions = predict(
    model_path="models/best_model.pkl",
    preprocessor_path="models/preprocessor.pkl",
    data=new_data
)

print(predictions[['probability_bad', 'credit_score', 'predicted_credit_risk']].head())

# Save predictions
predictions.to_csv("predictions.csv", index=False)

## Custom Configuration

Modify `config/config.yaml` to customize:
- Data paths and split ratios
- Preprocessing options (imputation, scaling, encoding)
- Feature engineering parameters
- Model hyperparameters
- Hyperparameter tuning settings
- Evaluation metrics
- Output paths

In [ ]:
# View current configuration
from src.config import config
import yaml

with open("../config/config.yaml", 'r') as f:
    print(f.read())

## CLI Usage

```bash
# Run with defaults
python main.py

# Run with custom data
python main.py --data data/raw/credit_data.csv

# Run with hyperparameter tuning
python main.py --tune-hyperparams --models logistic_regression random_forest xgboost lightgbm

# Run with custom config
python main.py --config config/custom_config.yaml --output results/experiment_1

# Make predictions
python main.py --predict data/raw/new_data.csv --model models/best_model.pkl --preprocessor models/preprocessor.pkl
```

## Features

### Data Preprocessing
- Missing value imputation (median for numerical, mode for categorical)
- Outlier detection and capping (IQR, Z-score methods)
- Feature scaling (Standard, Robust, MinMax)
- Categorical encoding (Weight of Evidence, One-Hot)
- Class imbalance handling (SMOTE, ADASYN, BorderlineSMOTE, SMOTEENN, SMOTETomek)
- Feature selection (Mutual Information, F-classif)

### Feature Engineering
- Credit-specific derived features (credit utilization, debt ratios, etc.)
- Interaction features
- Ratio features
- Binned features
- Weight of Evidence (WoE) encoding with Information Value calculation

### Models Supported
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- XGBoost
- LightGBM
- CatBoost

### Evaluation Metrics
- Accuracy, Precision, Recall, F1-Score
- ROC-AUC, PR-AUC
- Brier Score, Log Loss
- Matthews Correlation Coefficient
- Cohen's Kappa
- Kolmogorov-Smirnov (KS) Statistic
- Gini Coefficient
- Specificity, NPV, FPR, FNR
- Threshold optimization (F1, Youden's J, Precision, Recall)
- Calibration plots
- Confusion matrices
- Feature importance plots
- Score distribution plots

### Hyperparameter Tuning
- Optuna with TPE sampler
- Configurable search spaces
- Cross-validation based evaluation
- Timeout control

### Outputs
- Trained model (joblib)
- Preprocessor pipeline (joblib)
- Evaluation metrics (JSON)
- Comparison plots (PNG)
- Text reports
- Prediction CSV for new data